In [0]:
!pip install openpyxl

In [0]:
import pandas as pd
import glob
import re
from pyspark.sql.functions import col, datediff, when, concat, coalesce, lit, substring, date_add, date_format

In [0]:
def extract_date_from_filename(filename):
    """Extract date from filename in format 'dd.mm.yyyy'."""
    match = re.search(r'(\d{2})\.(\d{2})\.(\d{4})\.xlsx', filename)
    if match:
        day, month, year = match.groups()
        return f'{year}-{month}-{day}'  # Format as YYYY-MM-DD
    return None

def add_reporting_date(file_paths):
    # Create an empty list to store DataFrames
    dfs = []

    for file in file_paths:
        # Extract the filename
        filename = file.split('/')[-1]
        
        # Read the Excel file
        df = pd.read_excel(file, dtype=str)
        
        # Extract date from filename for all files
        reporting_date = extract_date_from_filename(filename)
        df['Reporting Date'] = reporting_date
        
        dfs.append(df)
    return dfs

In [0]:
# main path
source_path = "/dbfs/mnt/stppeedp/ppeedp/landing/data0/staging/eag/ey/ap_automation/"

# Define the path to write the output 
destination_path = f'dbfs:/mnt/stppeedp/ppeedp/prod/eag/ey/fdw/mfr/otp' 

pmt_path = source_path+'input_files/PMT.xlsx'
document_types_path = source_path+'input_files/Document_Types.xlsx'
rp_path = source_path+'input_files/RP.xlsx' 
direct_debit_path = source_path+'input_files/Direct_Debit.xlsx'
supplier_mapping_path = source_path+'input_files/Supplier Mapping.xlsx'

otp_paths = source_path+'otp/OTP*.xlsx' 
kz_paths = source_path+'otp/KZ*.xlsx' 


In [0]:
# Reading Static Tables - CompanyCode, Filter Advances

# Reading PMT filters and convert the Pandas DataFrame to a Spark DataFrame
pmt_df = pd.read_excel(pmt_path, dtype=str)
pmt = spark.createDataFrame(pmt_df)

# Reading Document types filter and convert the Pandas DataFrame to a Spark DataFrame
document_types_df = pd.read_excel(document_types_path, dtype=str)
document_types = spark.createDataFrame(document_types_df)

# Reading Dynamic Tables - RP, Direct Debit and Supplier Mapping

# Reading RP
rp_df = pd.read_excel(rp_path, dtype=str)
rp = spark.createDataFrame(rp_df)
rp = rp.dropDuplicates()

# Reading Direct Debit
direct_debit_df = pd.read_excel(direct_debit_path, dtype=str)
direct_debit = spark.createDataFrame(direct_debit_df)

# Reading Supplier Mapping 
supplier_mapping_df = pd.read_excel(supplier_mapping_path, dtype=str)
supplier_mapping = spark.createDataFrame(supplier_mapping_df)

# Define the path to your OTP Excel files
otp_file_paths = glob.glob(otp_paths)

# Add Reporting Date Column
dfs = add_reporting_date(otp_file_paths)

# Concatenate all DataFrames
otp_combined_df = pd.concat(dfs, ignore_index=True)

# Convert the Pandas DataFrame to a Spark DataFrame
input_otp = spark.createDataFrame(otp_combined_df)

# Define the path to your KZ Excel files
kz_file_paths = glob.glob(kz_paths)

# Use list comprehension to read and concatenate files
kz_combined_df = pd.concat([pd.read_excel(file,dtype=str) for file in kz_file_paths], ignore_index=True)

# Convert the Pandas DataFrame to a Spark DataFrame
input_kz = spark.createDataFrame(kz_combined_df)


In [0]:
# Adding columns
input_otp = input_otp.withColumn(
    "Clearing Entry Date - Net Due Date", datediff(col("Clearing Entry Date"), col("Net due date"))
).withColumn(
    "Late/OnTime", when(col("Clearing Entry Date - Net Due Date") <= 0, "On Time").otherwise("Late")
).withColumn(
    "ClearingDocument",
    substring(col("Clearing Document"), 1, 2)
)

# Join with PMT to get PMT Filter
input_otp = input_otp.join(pmt, input_otp.ClearingDocument == pmt.PMT_Id, "left").drop("PMT_Id","PMT_Description")

input_otp = input_otp.withColumnRenamed(
    "PMT_Flag","PMT Filter"
).withColumn(
    "PMT Filter",coalesce(col("PMT Filter"), lit(0))
)

# Join with document type to get Document Filter
input_otp = input_otp.join(document_types, input_otp["Document type"] == document_types["Document_Type"], "left").drop("Document_Type","Document_Description")

input_otp = input_otp.withColumnRenamed(
    "Document_Flag","Document Filter"
).withColumn(
    "Document Filter",coalesce(col("Document Filter"), lit(0))
)

# Concat with Company Code & Vendor
input_otp = input_otp.withColumn(
    "CC+Vendor", concat(col("Company Code"), col("Vendor"))
)

# Drop duplicates to keep only one row per CC_Code_Vendor
supplier_mapping_unique = supplier_mapping.dropDuplicates(["CC Code+Vendor"])

# Join with supplier_mapping_unique to get Vendor_Mapping
input_otp=input_otp.join(supplier_mapping_unique,input_otp["CC+Vendor"]==supplier_mapping_unique["CC Code+Vendor"], "left")\
    .drop(supplier_mapping_unique.Vendor).drop(supplier_mapping_unique["Company Code"])\
    .drop("CC Code+Vendor","Team Lead","Vendor type","Category","Vendor Status")\
    .withColumnRenamed("Manager","Vendor Mapping")\
    .withColumn("Vendor Mapping",coalesce(col("Vendor Mapping"), lit('Others')))

# Join with rp to get RP Filter
input_otp = input_otp.join(rp, input_otp["Vendor"] == rp["Vendor"], "left").drop(rp.Vendor)

input_otp = input_otp.withColumnRenamed(
    "Category","RP"
).withColumn(
    "RP",coalesce(col("RP"), lit(0))
)

# Drop duplicates to keep only one row per Clearing Document
input_kz_unique = input_kz.dropDuplicates(["Clearing Document"]).select('Clearing Document','Document Header Text')

# Join with input_kz to get PMT Type
input_otp = input_otp.join(input_kz_unique, input_otp["Clearing Document"] == input_kz_unique["Clearing Document"], "left").drop(input_kz_unique["Clearing Document"])

input_otp = input_otp.withColumnRenamed(
    "Document Header Text","PMT Type"
)

# Drop duplicates to keep only one row per Vendor
direct_debit_unique = direct_debit.dropDuplicates(["Vendor#"])

# Join with Direct Debit to get Direct Debit Vendor List
input_otp = input_otp.join(direct_debit_unique, input_otp["Vendor"] == direct_debit_unique["Vendor#"], "left").drop(direct_debit_unique['Vendor#']).drop(direct_debit_unique["FIXED/ VARIABLE"])

input_otp = input_otp.withColumnRenamed(
    "DD Vendor","Direct Debit Vendor List"
).withColumn(
    "Direct Debit Vendor List",coalesce(col("Direct Debit Vendor List"), lit(0))
).withColumn(
    'Direct Debit = W', when(col('PMT Type').contains('-W'), 1).otherwise(0)
).withColumn(
    "Rev Due Date less Doc Date", datediff(col("Net due date"), col("Document Date"))
).withColumn(
    'Revised Due Date',
    when(col('Rev Due Date less Doc Date') <= 7, date_add(col('Document Date'), 7))
    .otherwise(col('Net due date'))
).withColumn('Revised Due Date', date_format(col('Revised Due Date'), 'yyyy-MM-dd')
).withColumn(
    "Rev Clearing Entry Date  - Due Date", datediff(col("Clearing Entry Date"), col("Revised Due Date"))
).withColumn(
    'Rev Late/OnTime',
    when(col('Direct Debit = W') == 1, "On Time")
    .when(col('Rev Clearing Entry Date  - Due Date') <= 0, "On Time")
    .otherwise("Late")
)
# Write the data to the destination
input_otp.write.mode("overwrite").parquet(destination_path)
print(f"Data has been successfully processed and written at {destination_path}")